# Persistence and Checkpointers

**What you will build:** durable memory for a graph. In 0.6 and 1.4 *you* carried the message list forward by hand. LangGraph does it for you with a **checkpointer**: attach one, give each conversation a `thread_id`, and the graph remembers automatically — even across separate calls.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/23_persistence_and_checkpointers.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — LangChain + LangGraph on OpenRouter. Run once.
!pip install -q "langchain>=1.3,<2.0" "langgraph>=1.2,<2.0" "langchain-openai>=1.3,<2.0"

import os, getpass
from langchain_openai import ChatOpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
llm = ChatOpenAI(model=MODEL_NAME, base_url="https://openrouter.ai/api/v1",
                 api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready:", MODEL_NAME)

## A chatbot graph, then make it remember

Start with a one-node chat graph that has *no* memory — each call is independent (the statelessness of 0.1).

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState

def chat(state: MessagesState) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}

builder = StateGraph(MessagesState)
builder.add_node("chat", chat)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

plain = builder.compile()   # no checkpointer -> no memory
plain.invoke({"messages": [{"role": "user", "content": "My name is Sam."}]})
r = plain.invoke({"messages": [{"role": "user", "content": "What's my name?"}]})
print("no memory:", r["messages"][-1].content)   # it doesn't know

## Add a checkpointer

Compile the *same* graph with an `InMemorySaver`. Now every call tagged with the same `thread_id` shares state — the checkpointer saves the messages after each turn and reloads them on the next.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_graph = builder.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "conversation-1"}}
memory_graph.invoke({"messages": [{"role": "user", "content": "My name is Sam."}]}, config=cfg)
r = memory_graph.invoke({"messages": [{"role": "user", "content": "What's my name?"}]}, config=cfg)
print("with memory:", r["messages"][-1].content)   # now it knows

Same graph, same code — the only difference is the checkpointer and the `thread_id`. Persistence is a property you *attach*, not something you thread through every function. That's a real step up from carrying `message_history` by hand.

## Threads are separate conversations

A different `thread_id` is a clean slate — this is how one app serves many users without their chats mixing.

In [ ]:
other = {"configurable": {"thread_id": "conversation-2"}}
r = memory_graph.invoke({"messages": [{"role": "user", "content": "What's my name?"}]}, config=other)
print("different thread:", r["messages"][-1].content)   # no idea - separate conversation

## Time travel, executed

Every step of every thread got checkpointed — which means the past is *addressable*. First give the conversation one more turn so there's history worth rewinding, then list the checkpoints:

In [ ]:
memory_graph.invoke({"messages": [{"role": "user", "content": "My favourite food is pizza."}]},
                   config=cfg)

history = list(memory_graph.get_state_history(cfg))     # newest first
for i, snap in enumerate(history):
    n = len(snap.values.get("messages", []))
    print(f"checkpoint {i}: {n} messages")

Now the move that nothing in Blocks 0–1 could do: **resume from the past**. Pick the checkpoint right after the first exchange — when the graph knew Sam's name but not the pizza — and ask a question *from there*:

In [ ]:
past = next(s for s in history if len(s.values.get("messages", [])) == 2)   # after turn 1

r = memory_graph.invoke(
    {"messages": [{"role": "user", "content": "What do you know about me so far?"}]},
    config=past.config,                     # <- the past checkpoint's config, not cfg
)
print("answer from the past:", r["messages"][-1].content)

# and the ORIGINAL timeline is untouched:
now = memory_graph.get_state(cfg)
print("\nmain timeline still has", len(now.values["messages"]), "messages (pizza included)")

The answer knows Sam and *not* the pizza — you just ran a counterfactual from an earlier state, and it **forked**: the original timeline is intact, the what-if lives on its own branch. Three uses you'll actually reach for: reproducing bugs ("what did the agent know at step N?"), testing a prompt change against a mid-conversation state without replaying the whole chat, and building undo. This is the feature that has no Block 0 equivalent — you'd have had to design your whole storage model around it.

## Durable for real: `SqliteSaver`

`InMemorySaver` dies with the kernel. The production claim — "swap the checkpointer, change nothing else" — deserves a demonstration, not a footnote. Save a conversation to a SQLite *file*, then simulate a process restart by building a **brand-new** graph over the same file:

In [ ]:
!pip install -q "langgraph-checkpoint-sqlite>=2.0,<4.0"

import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

conn = sqlite3.connect("conversations.db", check_same_thread=False)
durable = builder.compile(checkpointer=SqliteSaver(conn))

cfg_d = {"configurable": {"thread_id": "sam-durable"}}
durable.invoke({"messages": [{"role": "user", "content": "My name is Sam and I live in Bilbao."}]},
               config=cfg_d)

# ---- pretend the process died: fresh connection, fresh graph object ----
conn2 = sqlite3.connect("conversations.db", check_same_thread=False)
durable_after_restart = builder.compile(checkpointer=SqliteSaver(conn2))

r = durable_after_restart.invoke(
    {"messages": [{"role": "user", "content": "Where do I live?"}]}, config=cfg_d)
print("after 'restart':", r["messages"][-1].content)

The second graph object shares nothing with the first except the file on disk — and the conversation survived. Your nodes didn't change by a character; persistence really is a property you attach. `PostgresSaver` is the same one-line swap for a real database. Compare with 1.4, where *you* wrote the save/load with `ModelMessagesTypeAdapter`: same capability, now with per-step granularity (that's what made time travel possible) and zero code in your functions.

::::{dropdown} 🔍 The checkpoint model, in one breath
:color: info

A **thread** is a sequence of **checkpoints**; each checkpoint is the full state after one super-step. `get_state(cfg)` reads the newest; `get_state_history(cfg)` walks them all (newest first); invoking with a past checkpoint's `config` forks from it. Savers (`InMemorySaver`/`SqliteSaver`/`PostgresSaver`) only decide *where* checkpoints live, never their shape — which is why swapping them touches no node code. If you want to *see* threads and checkpoints in a UI, LangGraph Studio visualizes exactly these objects over your graph.
::::

::::{dropdown} 🔧 Common issues
:color: info

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| **The agent doesn't remember** | No checkpointer, or a new `thread_id` each call | Compile with a checkpointer; reuse the same `thread_id` per conversation |
| **All users share one memory** | Same `thread_id` for everyone | Give each user/conversation its own `thread_id` |
| **Memory lost on kernel restart** | `InMemorySaver` is RAM-only | Use `SqliteSaver`/`PostgresSaver` for durable storage (demonstrated above) |
| `sqlite3` thread errors in a notebook | SQLite's default same-thread check | Pass `check_same_thread=False` to `sqlite3.connect` (as above) |
| Rewinding "loses" newer turns | It doesn't — resuming from a past checkpoint forks a branch | The original thread is intact; `get_state(cfg)` proves it |
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Read a snapshot.** Print `memory_graph.get_state(cfg).values["messages"]` and match it against the conversation so far. **Done when:** you can say where the system prompt is (trick question — this graph never set one; that's visible too).
2. **Fork on purpose.** From the same `past` checkpoint, send a *different* second turn ("My favourite food is sushi."), then ask each timeline what your favourite food is. **Done when:** pizza on one branch, sushi on the other, and you can explain why neither sees the other.
3. **Kill the process for real.** Restart the kernel (really), re-run only the setup + graph-definition + SqliteSaver cells, and ask `sam-durable` where Sam lives. **Done when:** it answers from disk — the only memory in this notebook that survives an actual restart.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**2 — Fork:**

```python
fork = memory_graph.invoke(
    {"messages": [{"role": "user", "content": "My favourite food is sushi."}]},
    config=past.config)

# each branch answers from its own history:
r1 = memory_graph.invoke({"messages": [{"role": "user", "content": "What's my favourite food?"}]},
                         config=cfg)                    # main timeline -> pizza
# for the fork, resume from ITS newest checkpoint: grab it from get_state_history
fork_cfg = memory_graph.get_state_history(past.config).__next__().config
r2 = memory_graph.invoke({"messages": [{"role": "user", "content": "What's my favourite food?"}]},
                         config=fork_cfg)               # fork -> sushi
```

Each invoke appended to its own branch of the checkpoint tree. Threads aren't lines — they're trees, and `thread_id` names the tree while `checkpoint_id` names the node. (If the fork bookkeeping feels fiddly here, that's fair — it's the one part where a UI like Studio genuinely helps.)

**3 —** The point of doing it *really*: `InMemorySaver` conversations and every Python object are gone, and `sam-durable` still answers. Persistence claims are only proven by an actual restart — the same "trust, but verify outside the agent" instinct as 0.7.
::::

**What's next:** in **2.4** we use the checkpointer for **human-in-the-loop** — pausing a graph mid-run to wait for a person, then resuming.